# External Tool Calling

This notebook focuses on external tool calling with plain LangChain tools. It reuses the agent runtime from notebook 1, but the tools now call external HTTP APIs instead of local Python-only functions.

## Learning Objectives

At the end of this notebook, you should be able to:

- Wrap external HTTP APIs as small `@tool` functions with their own error handling.
- Wrap a Python REPL and a Wikipedia search as `@tool` functions by calling the underlying libraries directly.
- Inspect how external tool calls appear in the agent's message trace.
- Compare plain `@tool` wrappers with the MCP approach from notebook 4.

## External tool calling in context

External tool calling means the agent can ask application code to fetch information or perform an action outside the model. The model chooses the tool and arguments; Python performs the external call.

```mermaid
flowchart LR
    U["User request"] --> A["Agent runtime"]
    A --> M["Model"]
    M -->|tool call| T["Python @tool wrapper"]
    T -->|HTTP request| API["External API"]
    API --> T
    T -->|ToolMessage| M
    M --> R["Final response"]
```

The important boundary is that the model does not make the HTTP request itself. The tool wrapper owns the network call, response parsing, timeout handling, and output shape.


## Imports

The custom HTTP tools use only standard-library networking utilities. The notebook also builds two more `@tool` wrappers: a Python REPL for deterministic calculations and a Wikipedia search for concise encyclopedic lookups, each calling the underlying library directly.

In [ ]:
import ast
import json
from typing import Any, cast
from urllib.error import HTTPError, URLError
from urllib.parse import quote
from urllib.request import Request, urlopen

import wikipedia as wikipedia_client
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from wikipedia.exceptions import WikipediaException

## Model setup

The model setup matches the earlier notebooks. The difference is in the tools: they now retrieve external data before the model writes the final response.


In [ ]:
load_dotenv(".env")

model = init_chat_model("groq:openai/gpt-oss-20b", temperature=0)
model

## A small HTTP helper

External tools should be defensive. The helper below sets a user agent, applies a timeout, parses JSON, and converts network failures into compact error dictionaries instead of raising through the notebook.


The model should not receive raw networking details. A compact helper gives every external tool the same behavior for timeouts, headers, JSON parsing, and error reporting. That keeps each tool focused on domain-specific response shaping.


In [ ]:
def request_json(url: str) -> dict[str, Any] | list[Any]:
    # Most public APIs expect a user agent and should not be called without a timeout.
    request = Request(
        url,
        headers={"User-Agent": "ds-ai-agent-external-tool-notebook/1.0"},
    )
    try:
        with urlopen(request, timeout=10) as response:
            payload = response.read().decode("utf-8")
            return cast(dict[str, Any] | list[Any], json.loads(payload))
    # Return a compact error object so the agent can explain the failure gracefully.
    except (HTTPError, URLError, TimeoutError, json.JSONDecodeError) as exc:
        return {"error": f"{type(exc).__name__}: {exc}"}

## Tool 1: package metadata from PyPI

This tool calls the public PyPI JSON API and returns only a compact subset of fields. Compact tool outputs keep the model context smaller and reduce irrelevant detail in the trace.


In [ ]:
@tool
def pypi_package_metadata(package_name: str) -> dict[str, str]:
    """Fetch compact package metadata from the public PyPI JSON API."""
    data = request_json(f"https://pypi.org/pypi/{quote(package_name)}/json")
    if not isinstance(data, dict):
        return {"error": "Unexpected response"}
    if "error" in data:
        return {"error": str(data.get("error", "Unexpected response"))}

    info = data.get("info", {})
    if not isinstance(info, dict):
        return {"error": "Missing package info"}

    return {
        "name": str(info.get("name", "")),
        "version": str(info.get("version", "")),
        "summary": str(info.get("summary", "")),
        "project_url": str(info.get("project_url", "")),
    }

## Tool 2: country profile from Open-Meteo

This tool calls the Open-Meteo geocoding API and returns a compact, normalised profile (population, timezone, and coordinates) instead of the full response.

In [ ]:
@tool
def country_profile(country_name: str) -> dict[str, str]:
    """Fetch a compact country profile from the Open-Meteo geocoding API."""
    url = (
        "https://geocoding-api.open-meteo.com/v1/search"
        f"?name={quote(country_name)}&count=1&language=en&format=json"
    )
    data = request_json(url)
    if not isinstance(data, dict):
        return {"error": "Unexpected response"}
    if "error" in data:
        return {"error": str(data["error"])}

    results = data.get("results")
    if not results or not isinstance(results[0], dict):
        return {"error": "No country profile found"}

    first = results[0]
    return {
        "name": str(first.get("name", country_name)),
        "country": str(first.get("country", "")),
        "population": str(first.get("population", "")),
        "timezone": str(first.get("timezone", "")),
        "coordinates": f"{first.get('latitude', '')}, {first.get('longitude', '')}",
    }

## Direct tool calls

Calling the tools directly verifies that the wrappers work independently of the agent runtime. This is the fastest way to debug external API issues because it removes model behavior from the equation.


In [ ]:
pypi_package_metadata.invoke({"package_name": "langchain"})

Calling the tool directly returns a compact dictionary: `langchain` at version 1.3.11 with a one-line summary and its project URL. The wrapper deliberately keeps only a few fields, so the model receives a small, relevant payload instead of the full PyPI response. Testing a tool directly like this, before handing it to an agent, isolates API and parsing problems from model behaviour.

In [ ]:
country_profile.invoke({"country_name": "Japan"})

The tool returned a compact profile for Japan: population 126,529,100, timezone `Asia/Tokyo`, and coordinates `35.68536, 139.7531`. The wrapper kept only these few fields and normalised them into a small dictionary, so the model receives a clean payload rather than the full geocoding response. Testing a tool directly like this, before handing it to an agent, isolates API and parsing problems from model behaviour.

## Tool 3: a restricted calculator

Code execution is powerful, but an unrestricted Python REPL would also let model-generated code read secrets, access files, call the network, or change the local environment. For this lesson, the tool accepts only arithmetic expressions and evaluates a small allowlist of AST nodes.

This still demonstrates deterministic tool use while keeping the capability proportionate to the task. Production systems should apply the same principle: expose the narrowest tool that meets the need, and isolate genuine code execution in a sandbox without application secrets.

In [ ]:
def evaluate_arithmetic(node: ast.expr) -> int | float:
    """Evaluate numeric literals and a small allowlist of arithmetic operators."""
    if isinstance(node, ast.Constant):
        value = node.value
        if isinstance(value, (int, float)) and not isinstance(value, bool):
            return value
        raise ValueError("Only numeric literals are allowed.")

    if isinstance(node, ast.UnaryOp):
        operand = evaluate_arithmetic(node.operand)
        if isinstance(node.op, ast.UAdd):
            return operand
        if isinstance(node.op, ast.USub):
            return -operand
        raise ValueError("That unary operator is not allowed.")

    if isinstance(node, ast.BinOp):
        left = evaluate_arithmetic(node.left)
        right = evaluate_arithmetic(node.right)
        if isinstance(node.op, ast.Add):
            return left + right
        if isinstance(node.op, ast.Sub):
            return left - right
        if isinstance(node.op, ast.Mult):
            return left * right
        if isinstance(node.op, ast.Div):
            return left / right
        if isinstance(node.op, ast.FloorDiv):
            return left // right
        if isinstance(node.op, ast.Mod):
            return left % right
        raise ValueError("That binary operator is not allowed.")

    raise ValueError("Only arithmetic expressions are allowed.")


@tool
def calculator(expression: str) -> str:
    """Evaluate a basic arithmetic expression without executing Python code."""
    if len(expression) > 100:
        return "ValueError: expression is too long."
    try:
        tree = ast.parse(expression, mode="eval")
        return str(evaluate_arithmetic(tree.body))
    except (SyntaxError, TypeError, ValueError, ZeroDivisionError) as exc:
        return f"{type(exc).__name__}: {exc}"


{"name": calculator.name, "description": calculator.description}

In [ ]:
calculator.invoke("17 * 19")

The calculator parsed `17 * 19` and returned `323`. It remains exact for the supported arithmetic operations, but unlike a Python REPL it rejects names, function calls, imports, file access, and other executable code.

## Tool 4: Wikipedia search

This tool wraps the `wikipedia` library behind the same tool interface used by the custom HTTP tools and the restricted calculator. The wrapper keeps results compact by returning only the top page and trimming the summary, which makes traces easier to inspect.

In [ ]:
# Wikipedia rate-limits the library's shared default User-Agent, so set a descriptive one
# (their API policy asks clients to identify themselves).
wikipedia_client.set_user_agent(
    "ds-ai-agent/1.0 (neuefische bootcamp; educational use)"
)


@tool
def wikipedia_search(query: str) -> str:
    """Search Wikipedia and return the top page title with a short summary."""
    try:
        titles = wikipedia_client.search(query, results=1)
        if not titles:
            return f"No Wikipedia page found for {query!r}."
        page = wikipedia_client.page(titles[0], auto_suggest=False)
        # Keep the output compact so the agent trace stays easy to inspect.
        return f"Page: {page.title}\nSummary: {page.summary[:500]}"
    except WikipediaException as exc:
        return f"{type(exc).__name__}: {exc}"


{"name": wikipedia_search.name, "description": wikipedia_search.description}

In [ ]:
wikipedia_search.invoke("LangChain")

The tool returned the LangChain page title and a short summary, trimmed by the compact settings (one result, 500 characters). Note the `set_user_agent(...)` call in the previous cell: the `wikipedia` library's shared default User-Agent is rate-limited by Wikipedia (HTTP 429), so external tools should identify themselves with a descriptive User-Agent to stay within a service's policy.

## Build the external-tool agent

The agent receives all four tools through the same `tools` list. The runtime decides whether to call one tool, several tools, or no tools based on the request.

Custom `@tool` functions are the simplest approach when a tool belongs to one application, or when you need to wrap a well-known library such as a restricted calculator or Wikipedia yourself. MCP becomes useful when the same capabilities should be exposed as a separate server, reused by multiple clients, or maintained independently from the notebook code.

In [ ]:
external_tools = [pypi_package_metadata, country_profile, calculator, wikipedia_search]

external_agent = create_agent(
    model=model,
    tools=external_tools,
    system_prompt=(
        "Use external tools for package facts, country facts, concise encyclopedia lookups, or small calculations. "
        "Keep the final answer concise and mention if a tool returned an error."
    ),
)

## Run an agent request

The request below uses external data, a Wikipedia lookup, and a small calculation. The agent can choose from all available tools and combine their results in one final response.

In [ ]:
external_result = external_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Use the available external tools to look up compact metadata for "
                    "the langchain package and a compact country profile for Japan. "
                    "Also find one concise Wikipedia fact about LangChain and calculate 17 * 19 exactly. Return four short bullets."
                ),
            }
        ]
    }
)

external_result["messages"][-1].content

Given one request, the agent chose the tools it needed and combined their results into four bullets: the langchain package metadata (version 1.3.11), the country profile for Japan (population 126,529,100, timezone Asia/Tokyo), one Wikipedia fact about LangChain, and the calculation `17 * 19 = 323`. The runtime decided which tools to call and in what order, just as in notebook 1, but here the work reached out to external services.

## Inspect the trace

External tool calls appear in the same message trace as local tool calls. The trace shows which tool was requested, which arguments were generated, and what tool messages came back into the conversation.


In [ ]:
def summarize_messages(messages):
    rows = []
    for index, message in enumerate(messages):
        rows.append(
            {
                "index": index,
                "type": type(message).__name__,
                "content": getattr(message, "content", None),
                "tool_calls": getattr(message, "tool_calls", None),
            }
        )
    return rows

In [ ]:
summarize_messages(external_result["messages"])

## Design notes for external tools

- Keep tool inputs narrow and explicit. A clear schema helps the model choose the right arguments.
- Keep tool outputs compact. Large raw API payloads increase context size and make traces harder to inspect.
- Handle timeouts and malformed responses inside the tool wrapper. The agent should receive a controlled error message, not a stack trace.
- Test tools directly before adding them to an agent. Direct calls make API and parsing problems easier to isolate.
- Treat execution tools carefully. A REPL can run arbitrary code, so production systems should restrict access, inputs, permissions, and runtime environment.
- Prefer compact retrieval settings for broad knowledge tools. Wikipedia results can become long, so limit result count and content length for notebook traces.
- Use MCP when external capabilities should be shared across many clients or maintained as separate services. Use plain `@tool` wrappers when the integration is small and application-specific.


## Summary

In this notebook you:

- Wrapped external HTTP APIs (PyPI and Open-Meteo) as `@tool` functions with compact output and error handling.
- Wrapped a Python REPL and a Wikipedia search as `@tool` functions too, and combined all four in one agent.
- Inspected the message trace to see external tool calls and their arguments.
- Compared plain `@tool` wrappers with MCP: use wrappers for small, application-specific integrations, and MCP when a capability should be shared across clients.

## References & Further Reading

- [**LangChain Agents**](https://docs.langchain.com/oss/python/langchain/agents): The agent runtime that calls these tools.
- [**Tools**](https://docs.langchain.com/oss/python/langchain/tools): Defining and using LangChain tools.
- [**Python REPL integration**](https://docs.langchain.com/oss/python/integrations/tools/python): LangChain's guide to running Python as a tool.
- [**LangGraph Overview**](https://docs.langchain.com/oss/python/langgraph/overview): The runtime agents are built on.